In [76]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

In [77]:
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [78]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [79]:
userItemData = pd.read_csv("tech_purchases.csv")
userItemData.head(3)

,userId,ItemId,Name,Category,Salesperson
0,1001,5001,Samsung A21,Hardware,Sally
1,1001,5002,Scase,Accessories,Sally
2,1001,5004,Insurance,Extra,Sally


In [80]:
userItemData.drop(['Category','Salesperson'],axis = 1).head(2)

,userId,ItemId,Name
0,1001,5001,Samsung A21
1,1001,5002,Scase


In [81]:
from sklearn.metrics.pairwise import cosine_similarity

# Build binary user-item matrix
user_item_matrix = (
    userItemData.assign(purchased=1)
    .pivot_table(index='userId', columns='Name', values='purchased', fill_value=0)
)

# Item-item affinity (cosine similarity)
item_similarity = pd.DataFrame(
    cosine_similarity(user_item_matrix.T),
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

def recommend_items(item_name, top_n=3):
    """Recommend items with highest affinity to the given item."""
    if item_name not in item_similarity.index:
        raise ValueError(f"Item '{item_name}' not found")
    scores = item_similarity[item_name].drop(item_name).sort_values(ascending=False)
    return scores.head(top_n)

def recommend_for_user(user_id, top_n=3):
    """Recommend unpurchased items based on affinity to the user's purchased items."""
    if user_id not in user_item_matrix.index:
        raise ValueError(f"User '{user_id}' not found")
    purchased = user_item_matrix.loc[user_id]
    purchased_items = purchased[purchased > 0].index
    scores = item_similarity[purchased_items].sum(axis=1)
    scores = scores.drop(purchased_items).sort_values(ascending=False)
    return scores.head(top_n)

print(recommend_items('Samsung A21'))
print(recommend_for_user(1003))

Name
Scase        0.666667
Insurance    0.577350
Apple 12     0.000000
Name: Samsung A21, dtype: float64
Name
Insurance        1.154701
Apple 12         0.000000
Lighting Cord    0.000000
dtype: float64


#print out a affinity rating list with percentage using nlargest.

In [82]:
top_items = item_similarity[item].drop(item).nlargest(3).mul(100)

print(f"Items most similar to {item}:")
for name, score in top_items.items():
    print(f"{name}: {score:.1f}%")

Items most similar to iCase:
Lighting Cord: 81.6%
Apple 12: 50.0%
Insurance: 35.4%


In [83]:
for item in item_similarity.columns:
    top_items = (
        item_similarity[item]
        .drop(item)
        .nlargest(3)
        .mul(100)
    )
    print(f"\nItems similar to {item}:")
    print(top_items.map(lambda score: f"{score:.1f}%"))


Items similar to Apple 12:
Name
Lighting Cord    81.6%
iCase            50.0%
Insurance         0.0%
Name: Apple 12, dtype: str

Items similar to Insurance:
Name
Samsung A21    57.7%
Scase          57.7%
iCase          35.4%
Name: Insurance, dtype: str

Items similar to Lighting Cord:
Name
Apple 12     81.6%
iCase        81.6%
Insurance    28.9%
Name: Lighting Cord, dtype: str

Items similar to Samsung A21:
Name
Scase        66.7%
Insurance    57.7%
Apple 12      0.0%
Name: Samsung A21, dtype: str

Items similar to Scase:
Name
Samsung A21    66.7%
Insurance      57.7%
Apple 12        0.0%
Name: Scase, dtype: str

Items similar to iCase:
Name
Lighting Cord    81.6%
Apple 12         50.0%
Insurance        35.4%
Name: iCase, dtype: str


#build a recommendation list as an input form for one item related to all other items.

In [84]:
recommend_item_dropdown = widgets.Dropdown(
    options=item_similarity.index.tolist(),
    value=item if item in item_similarity.index else item_similarity.index[0],
    description="Item:"
)

def show_item_recommendations(selected_item):
    scores = (
        item_similarity[selected_item]
        .drop(selected_item)
        .sort_values(ascending=False)
        .head(3)
        .mul(100)
    )
    print(f"Items related to {selected_item}:")
    for name, score in scores.items():
        print(f"{name}: {score:.1f}%")

display(widgets.interactive(show_item_recommendations, selected_item=recommend_item_dropdown))

interactive(children=(Dropdown(description='Item:', index=5, options=('Apple 12', 'Insurance', 'Lighting Cord'…

In [85]:
def show_recommendations(selected_item):
    scores = (
        item_similarity[selected_item]
        .drop(selected_item)
        .sort_values(ascending=False)
        .head(3)
        .mul(100)
    )
    print(f"Items related to {selected_item}:")
    for name, score in scores.items():
        print(f"{name}: {score:.1f}%")

item_dropdown = widgets.Dropdown(
    options=item_similarity.index.tolist(),
    value=item if item in item_similarity.index else item_similarity.index[0],
    description="Item:",
)

display(widgets.interactive(show_recommendations, selected_item=item_dropdown))

interactive(children=(Dropdown(description='Item:', index=5, options=('Apple 12', 'Insurance', 'Lighting Cord'…